In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

DEFAULTS = {
    "catalog_name": "proyectoFinal",
    "raw_schema": "raw",
    "bronze_schema": "bronze",
    "silver_schema": "silver",
    "gold_schema": "gold",
    "storage_account": "asaproyecto",
    "raw_container": "raw",
    "bronze_container": "bronze",
    "silver_container": "silver",
    "gold_container": "gold",
    "read_principal": "datareaders",
    "write_principal": "dataengineers",
    "storage_credential_name": "credential",
    "external_location_raw": "exlt_raw",
    "external_location_bronze": "exlt_bronze",
    "external_location_silver": "exlt_silver",
    "external_location_gold": "exlt_gold",
    "access_connector_or_mi_name": "ac-proyectoFinal",
    "bronze_table_prefix": "",
    "silver_table_prefix": "",
    "gold_table_prefix": "",
}

def _safe_get_widget(name: str, default_value: str) -> str:
    try:
        dbutils.widgets.text(name, default_value)
        return dbutils.widgets.get(name)
    except Exception:
        return default_value

catalog_name = _safe_get_widget("catalog_name", DEFAULTS["catalog_name"]).strip()
raw_schema = _safe_get_widget("raw_schema", DEFAULTS["raw_schema"]).strip()
bronze_schema = _safe_get_widget("bronze_schema", DEFAULTS["bronze_schema"]).strip()
silver_schema = _safe_get_widget("silver_schema", DEFAULTS["silver_schema"]).strip()
gold_schema = _safe_get_widget("gold_schema", DEFAULTS["gold_schema"]).strip()
storage_account = _safe_get_widget("storage_account", DEFAULTS["storage_account"]).strip()
raw_container = _safe_get_widget("raw_container", DEFAULTS["raw_container"]).strip()
bronze_container = _safe_get_widget("bronze_container", DEFAULTS["bronze_container"]).strip()
silver_container = _safe_get_widget("silver_container", DEFAULTS["silver_container"]).strip()
gold_container = _safe_get_widget("gold_container", DEFAULTS["gold_container"]).strip()
read_principal = _safe_get_widget("read_principal", DEFAULTS["read_principal"]).strip()
write_principal = _safe_get_widget("write_principal", DEFAULTS["write_principal"]).strip()
storage_credential_name = _safe_get_widget("storage_credential_name", DEFAULTS["storage_credential_name"]).strip()
external_location_raw = _safe_get_widget("external_location_raw", DEFAULTS["external_location_raw"]).strip()
external_location_bronze = _safe_get_widget("external_location_bronze", DEFAULTS["external_location_bronze"]).strip()
external_location_silver = _safe_get_widget("external_location_silver", DEFAULTS["external_location_silver"]).strip()
external_location_gold = _safe_get_widget("external_location_gold", DEFAULTS["external_location_gold"]).strip()
access_connector_or_mi_name = _safe_get_widget("access_connector_or_mi_name", DEFAULTS["access_connector_or_mi_name"]).strip()
bronze_table_prefix = _safe_get_widget("bronze_table_prefix", DEFAULTS["bronze_table_prefix"]).strip()
silver_table_prefix = _safe_get_widget("silver_table_prefix", DEFAULTS["silver_table_prefix"]).strip()
gold_table_prefix = _safe_get_widget("gold_table_prefix", DEFAULTS["gold_table_prefix"]).strip()

raw_uri = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net"
bronze_uri = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net"
silver_uri = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net"
gold_uri = f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net"

bronze_catalog = f"{catalog_name}.{bronze_schema}"
silver_catalog = f"{catalog_name}.{silver_schema}"
gold_catalog = f"{catalog_name}.{gold_schema}"

source_files = {
    "movies": f"{raw_uri}/Movies.csv",
    "film_details": f"{raw_uri}/FilmDetails.csv",
    "more_info": f"{raw_uri}/MoreInfo.csv",
    "poster_path": f"{raw_uri}/PosterPath.csv",
}

bronze_tables = {
    "movies": f"{bronze_catalog}.{bronze_table_prefix}movies",
    "film_details": f"{bronze_catalog}.{bronze_table_prefix}film_details",
    "more_info": f"{bronze_catalog}.{bronze_table_prefix}more_info",
    "poster_path": f"{bronze_catalog}.{bronze_table_prefix}poster_path",
}

silver_tables = {
    "movies_silver": f"{silver_catalog}.{silver_table_prefix}movies_silver",
    "film_details_silver": f"{silver_catalog}.{silver_table_prefix}film_details_silver",
    "more_info_silver": f"{silver_catalog}.{silver_table_prefix}more_info_silver",
    "poster_silver": f"{silver_catalog}.{silver_table_prefix}poster_silver",
    "movie_conformed": f"{silver_catalog}.{silver_table_prefix}movie_conformed",
    "movie_genres_exploded": f"{silver_catalog}.{silver_table_prefix}movie_genres_exploded",
}

gold_tables = {
    "dim_director": f"{gold_catalog}.{gold_table_prefix}dim_director",
    "dim_language": f"{gold_catalog}.{gold_table_prefix}dim_language",
    "dim_date": f"{gold_catalog}.{gold_table_prefix}dim_date",
    "dim_genre": f"{gold_catalog}.{gold_table_prefix}dim_genre",
    "dim_movie": f"{gold_catalog}.{gold_table_prefix}dim_movie",
    "bridge_movie_genre": f"{gold_catalog}.{gold_table_prefix}bridge_movie_genre",
    "fact_movie_metrics": f"{gold_catalog}.{gold_table_prefix}fact_movie_metrics",
}

gold_views = {
    "vw_powerbi_movie_finance": f"{gold_catalog}.vw_powerbi_movie_finance",
    "vw_powerbi_movie_genre": f"{gold_catalog}.vw_powerbi_movie_genre",
}

AUDIT_COLUMNS = [
    "_source_file",
    "_ingestion_ts",
    "_load_date",
    "_record_hash",
    "_pipeline_run_ts",
]

def save_delta_table(df, full_table_name: str, mode: str = "overwrite"):
    (
        df.write.format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .option("mergeSchema", "true")
        .saveAsTable(full_table_name)
    )

def standardize_text(column):
    return F.when(column.isNull(), F.lit(None)).otherwise(
        F.trim(F.regexp_replace(column.cast("string"), r"\s+", " "))
    )

def to_snake_case(column_name: str) -> str:
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in column_name.strip().lower())
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned.strip("_")

def select_existing_audit_columns(df):
    return [F.col(c) for c in AUDIT_COLUMNS if c in df.columns]

def print_runtime_config():
    print("Configuración cargada:")
    print(f"  catalog_name={catalog_name}")
    print(f"  raw_uri={raw_uri}")
    print(f"  bronze_uri={bronze_uri}")
    print(f"  silver_uri={silver_uri}")
    print(f"  gold_uri={gold_uri}")
    print(f"  read_principal={read_principal}")
    print(f"  write_principal={write_principal}")
    print(f"  storage_credential_name={storage_credential_name}")
    print(f"  external_location_raw={external_location_raw}")

print_runtime_config()